# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [4]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25', 'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided information, the most common issues with loans appear to be:\n\n- Errors and discrepancies in loan balances and account information.\n- Difficulties in managing repayment, including being unable to direct extra payments toward principal or paying off smaller loans faster.\n- Bad or incorrect information on credit reports due to mismanagement or improper reporting.\n- Issues related to loan transfers without proper notification.\n- Problems with the handling of forbearances and the accumulation of interest leading to increased loan balances.\n- Poor communication and transparency from lenders or servicers.\n\nIn summary, the most common issue seems to be the mismanagement and misreporting of loan information, leading to incorrect balances, credit impacts, and difficulty in repayment management.'

In [11]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints were not handled in a timely manner. Specifically, at least two complaints by consumers noted that responses or resolutions took significantly longer than expected:\n\n- One complaint from a consumer filed on 03/28/25 (Complaint ID: 12709087) was marked as "Not timely" despite the company response being "Closed with explanation."\n- Another consumer complaint received on 04/24/25 (Complaint ID: 13160766) was marked as "Timely response?": "Yes," but the nature of the complaints indicates ongoing issues, suggesting delays in resolution or response times may have impacted their ability to get timely handling in some cases.\n\nAdditionally, in the complaint from 04/14/25 (Complaint ID: 12973003), the consumer indicated issues persisting over several weeks, and in another complaint from 04/05/25 (Complaint ID: 12832400), the consumer expressed that Aidvantage had failed to respond as required by policy.\n\nTherefore, while some complai

In [12]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily because they faced issues such as:\n\n1. **Inability to Afford Payments:** Many borrowers could not increase their monthly payments without sacrificing essential living expenses like bills, food, and transportation.\n\n2. **Accumulating Interest and Unmanageable Debt:** When loans were deferred or placed into forbearance, interest continued to accrue, often negating any payments made and causing the total debt to increase over time.\n\n3. **Lack of Clear or Adequate Communication:** Borrowers were often not properly notified about important details such as when repayments would resume, loan transfers between servicers, or changes in payment arrangements, leading to unintentional delinquency.\n\n4. **Complex or Opaque Loan Management:** Borrowers reported difficulties understanding their loan balances, interest calculations, and the effects of multiple transfers between loan servicers, which contributed to confusion and missed payments.\n

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be dealing with the lender or servicer, particularly related to misunderstandings or disputes over fees, payment application, loan information, and loan validity. Many complaints involve problems with how payments are handled, incorrect or bad information about loans, and issues with fees or charges that are not properly explained or justified.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that several complaints were responded to in a timely manner, with responses marked as "Yes" under the "Timely response?" field. Specifically, complaints with IDs 13197090, 12792958, 13160766, and 13410623 all received responses that were classified as timely. \n\nHowever, there are multiple complaints, particularly those involving serious ongoing issues and unresolved disputes, where the responses may not have adequately addressed the core problems. But according to the data, no complaints went completely unhandled or were responded to late.\n\nTherefore, the answer is: No, there is no evidence in the provided data indicating that any complaints did not get handled in a timely manner.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including misunderstandings or issues with their payment plans, miscommunication from loan servicers, problems with automatic payments, and difficulties in navigating the repayment processes. Specifically, some individuals experienced being incorrectly steered into wrong types of forbearances or had their autopayments discontinued without proper notification, leading to overdue payments and negative impacts on their credit scores. Others faced issues such as loans being transferred without their knowledge, emails going to spam, or not receiving timely responses from the servicers, which hindered their ability to stay current with payments. Overall, these failures often stem from administrative errors, lack of clear communication, and complex procedures that make it challenging for borrowers to remain in good standing.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.
##### ✅ Answer:
Example Query:<br>
Suppose a user wants to find complaints or documents that mention a specific IRS tax form: "Form 1098-E interest reporting"<br>
Justification:<br>
1) BM25 will excel at retrieving documents that mention "Form 1098-E" and "interest reporting" exactly, because it matches on the presence and frequency of these specific terms.<br>
2) Embeddings may not prioritize this document as highly, especially if "Form 1098-E" is a rare or out-of-vocabulary term for the embedding model, or if the model focuses more on the general concept of "interest reporting" rather than the specific form number.<br>
Why BM25 is better here:<br>
1) In the financial domain, queries often include unique identifiers, regulation numbers, or form names (e.g., "Form 1099-MISC", "Regulation D", "Section 401(k)"). BM25’s exact matching ensures that documents containing these rare, domain-specific terms are ranked at the top.<br>
2) Embedding-based retrievers might miss the importance of these exact terms, especially if they are not common in the model’s training data.<br><br>

When searching for documents that reference specific financial forms, codes, or regulations (like "Form 1098-E interest reporting"), BM25 is superior because it directly matches these rare, technical terms, ensuring precise retrieval. Embeddings may generalize too much and overlook the importance of the exact phrase.


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided information, a common issue with loans, particularly student loans, involves errors and mismanagement related to the handling of loan information. This includes issues such as errors in loan balances, misapplied payments, wrongful denials of payment plans, and inaccurate or incomplete data provided by lenders or servicers. Additionally, problems like incorrect information on credit reports, unauthorized transfers of loans, and privacy violations are frequently reported. Overall, a most common issue appears to be the mishandling or incorrect management of loan data and communication problems with lenders or servicers.\n'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

"Based on the provided information, it appears that at least one complaint did not get handled in a timely manner. Specifically, one complaint related to a student loan account review has been open for nearly 18 months without resolution, despite the complainant's repeated requests and follow-ups. The complaint notes that it has taken over a year for a response and resolution, indicating it was not addressed in a timely manner."

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. Lack of awareness and understanding: Some borrowers were not informed by their financial aid officers that they needed to repay their loans, leading to surprise and confusion about their repayment obligations.\n\n2. Poor communication and notification: Borrowers reported not receiving timely notifications about when payments were due, loan transfers, or changes in loan servicing, which resulted in missed payments or being unaware of their repayment status.\n\n3. Difficulty managing interest and payments: Many borrowers found that interest continued to accrue even when they were unable to make payments, which increased their overall debt and made repayment more difficult. Some options like forbearance or deferment led to more interest accumulation, further prolonging the time needed to pay off the loans.\n\n4. Financial hardship and economic factors: Borrowers faced challenges such as stagnant wages, inflation,

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issues with loans, based on the provided complaints, appear to be:\n\n- Errors in loan balances and interest calculations\n- Unauthorized or incorrect reporting of account status and delinquency\n- Difficulties in obtaining accurate information about loan terms, balances, and repayment plans\n- Problems with payments being misapplied, reversed, or not properly credited\n- Lack of transparency, communication, and proper notifications from servicers\n- Issues related to loan transfer, sale, or mismanagement by loan servicers\n- Inaccurate credit reporting and negative impact on credit scores\n- Problems with handling forbearance, deferments, and income-driven repayment plans\n- Unauthorized disclosure of personal information and violations of privacy laws\n\nIn summary, the most common and recurring issue is the mismanagement and inaccurate handling of loan information, leading to confusion, dispute, and financial hardship for borrowers.'

In [26]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints data, several complaints indicate that issues were not handled or responded to in a timely manner. Specifically, the following complaints explicitly mention delays or failures in timely response:\n\n- Complaint ID 13091395 (Maximus Federal Services, Inc., KY): The complaint notes "it is now over XXXX days later and the case is still marked \'In Review\' with no reply," indicating a delay beyond the expected timeframe.\n- Complaint ID 13070546 (Edfinancial Services, MO): The complainant states, "It has been over 1 year since I submitted this request and have not received a response," reflecting a failure to respond in a timely manner.\n- Complaint ID 13205525 (Nelnet, MI): The complainant reports, "over 30 days ago and still haven\'t gotten a response," indicating a delay beyond the lawful response timeline.\n- Complaint ID 13410623 (Maximus Federal Services, KY): The complaint states the original complaint was not addressed and the issue persists over 

In [27]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily due to a combination of factors such as lack of clear information about repayment options, being misled or steered into forbearance without understanding the accruing interest, income difficulties, and systemic issues within loan servicing systems. Many borrowers were not adequately informed about the true cost of forbearance—including how interest would continue to accumulate—leading to ballooning balances that made repayment unaffordable. Additionally, some borrowers faced challenges with the transfer of their loans to different servicers without proper notification, causing confusion and missed payments. Others encountered difficulties obtaining suitable repayment plans, including income-driven options, or were unaware of alternative forgiveness programs. Systemic issues like poor communication, improper handling of loan data, and lack of transparency contributed significantly to borrowers' inability to successfully repay their loans.

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.
##### ✅ Answer:
Generating multiple reformulations of a user query can improve recall by increasing the chances of retrieving all relevant documents, especially when different documents use varied language or terminology to describe the same concept.<br><br>
By generating and searching with multiple reformulations, the system increases the diversity of search queries, which helps surface more relevant documents that might otherwise be missed with a single query. This leads to higher recall, meaning more of the truly relevant documents are retrieved and available for downstream tasks like answer generation.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be related to problems with federal student loan servicing. Specific issues include errors in loan balances, misapplied payments, wrongful denials of payment plans, and disputes over loan interest rates and reporting. Additionally, there are concerns about incorrect or misleading credit reporting, unauthorized increases in interest rates, and failure to verify the legitimacy of debts.'

In [34]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided context, yes, there are complaints indicating that they did not get handled in a timely manner. Specifically, multiple complaints from consumers regarding delays or lack of responses include:\n\n- A complaint about their application not being processed despite multiple inquiries, with the confirmation that it would take 15 days for someone to reach out, and as of the date of the complaint, no contact had been made. The response was marked as "No" regarding timely response.\n- Another complaint about ongoing issues with loan payment processing and unacceptable wait times of four hours or more when trying to contact the company.\n- A complaint about a dispute settlement sent over 30 days ago that had not received a response, which was flagged as negligent and not in compliance with the law.\n\nTherefore, several complaints did not get handled in a timely manner.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"Based on the provided context, people failed to pay back their loans primarily due to several reasons:\n\n1. **Financial Hardship and Unemployment:** Many borrowers experienced severe financial difficulties or inability to secure employment after graduation, making it challenging to make regular loan payments. For example, one individual was unable to secure employment in their field, affecting their ability to repay loans.\n\n2. **Lack of Proper Information and Transparency:** Borrowers reported that they were not adequately informed about the long-term consequences of taking out loans, the status of their educational institution, or the procedures for repayment. For instance, one person indicated that their loan payments began before the grace period without explanation.\n\n3. **Problems with Loan Servicing and Management:** Some individuals faced issues like being reported delinquent while their cases were unresolved, or receiving inaccurate information about their debts. There wer

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [37]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the information provided, the most common issues with loans, particularly student loans, include:\n\n- Errors in loan balances, misapplied payments, and wrongful denials of payment plans.\n- Dealing with lenders or servicers regarding bad or inaccurate information about loans.\n- Trouble with how payments are being handled, such as restrictions on extra payments or predatory repayment practices.\n- Discrepancies and inaccuracies in loan data, balances, interest calculations, or reporting to credit bureaus.\n- Poor communication, lack of transparency, and failure to provide proper documentation like original signed Master Promissory Notes (MPNs).\n- Unauthorized transfer of loans without notice, leading to confusion and inaccuracies.\n- Problems with loan classification (e.g., mislabeling as a different loan type) or improper ending of deferments.\n- Issues related to loan forgiveness, discharge, or discharge-related mismanagement.\n\nOverall, a recurring theme is the mishandl

In [39]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints and responses, some complaints indicate delays or issues in handling in a timely manner. For example:\n\n- Complaint ID 12935889 (Mohela, 04/11/25): Response marked as "No" for timely response, indicating it was not handled promptly.\n- Complaint ID 12668396 (Mohela, 03/26/25): Response marked as "No," also indicating it was not handled in a timely manner.\n- Complaint ID 12709087 (MOHELA, 03/28/25): Response was "No" for timely response.\n- Complaint ID 12744910 (Maximus Federal Services, 04/01/25): Response marked "Yes" for timely response.\n- Complaint ID 12832400 (Maximus, 04/05/25): Response marked "Yes" for timely response.\n- Complaint ID 13410623 (Maximus, 05/08/25): Response marked "Yes" for timely response.\n- Complaint ID 13515083 (TransUnion, 05/14/25): Response marked "Yes" for timely response.\n\nFrom this, at least some complaints about repayment issues, account management, and investigations not being handled promptly are evident, speci

In [40]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for a variety of reasons, including:\n\n1. **Interest Accumulation and Compounding:** Many borrowers found that interest continued to accumulate and compound during deferment or forbearance periods, making the total debt grow over time despite their efforts to pay.\n\n2. **Lack of Clear Communication and Education:** Borrowers often reported being poorly informed about repayment options, such as income-driven repayment plans, loan forgiveness programs, or the risks of forbearance steering practices. This lack of information led to unintentional default or payments not being made correctly.\n\n3. **Inability to Afford Payments:** Many borrowers faced financial hardships, such as unemployment, low income, or unexpected expenses, which made it impossible to keep up with repayment schedules, especially when options to lower payments were limited or interest kept increasing the debt.\n\n4. **Mismanagement and Errors by Servicers:** Complaints indicate 

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [41]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [42]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [43]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [44]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [45]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [46]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints, the most common issues with loans appear to be related to miscommunication, improper handling of payments, incorrect reporting, and lack of transparency from loan servicers. Specifically, many complaints mention problems such as:\n\n- Difficulties with repayment plans and unexpected payment amounts.\n- Issues with auto-debit setups and discrepancies in scheduled payments.\n- Reporting errors including loans appearing in default or delinquency without proper cause.\n- Discrepancies or delays in loan servicing information, such as failure to notify of servicer changes.\n- Unauthorized collection activities and violations of privacy laws.\n- Challenges in obtaining clear information about loan status, balances, or documentation.\n\nWhile the most frequent issue seems to center around poor communication and errors in loan handling, a key theme is the improper or inconsistent management of loan information and repayment processes. \n\nTherefore, the most c

In [47]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, it appears that several complaints were marked as "Closed with explanation" but the responses often indicate that the issues were not handled in a timely manner or resulted in unresolved concerns. In particular, the complaint about Nelnet (row 17) describes multiple attempts to communicate serious misconduct, yet Nelnet never responded directly to the complaint, and the issues remain unaddressed. Similarly, the complaint from MOHELA (row 4) and EdFinancial Services (row 5) involved timely responses ("Yes" for timely response), but the nature of their responses suggests that issues such as errors and disputes were not satisfactorily resolved.\n\nSpecifically, the complaint about Nelnet (05/04/25 for transfer and misconduct) indicates that despite multiple letters and notices, the service provider did not respond to the complaint, which suggests that some complaints did not get handled in a timely manner. \n\nOverall, the evidence points to some complai

In [48]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including issues with the handling and transparency of loan information, problems with repayment plans, misreporting or outdated reporting of loan status, and alleged illegal or improper collection practices. Some borrowers experienced difficulties with accessing accurate information or with the administrative processes, which caused stress and confusion. Additionally, there are cases where borrowers dispute the legitimacy of their debt or claim that their personal and financial data was mishandled or compromised, leading to disputes about repayment obligations.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?
##### ✅ Answer:
With short, repetitive sentences, semantic chunking may create too many small, similar chunks. To address this, increase the minimum chunk size, raise the similarity threshold, or use fixed-size or structure-based chunking. This ensures each chunk contains enough context and reduces redundancy, improving retrieval quality and efficiency.

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [2]:
### Load Open AI API Key
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [204]:
### Load LangChain API Key and enable trace and name the LangChain Project
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

os.environ["LANGCHAIN_PROJECT"] = f"S09 - SDG - {uuid4().hex[0:8]}"

In [3]:
### Getting Live data(Financial) from ArxivQueryRun
from arxiv_fetcher import ArxivFetcher

arxiv_query = input("Enter an Arxiv search query to fetch relevant data: ").strip()

fetcher = ArxivFetcher()
financial_knowledge_resources = fetcher.fetch_and_process_papers(arxiv_query, max_results=3)
print(f"Loaded {len(financial_knowledge_resources)} document chunks from Arxiv papers.")

Searching Arxiv for papers about: financial analysis wealth and asset management
Processing paper: Monetary Policy and Wealth Inequalities in Great Britain: Assessing the role of unconventional policies for a decade of household data
Successfully processed paper: Monetary Policy and Wealth Inequalities in Great Britain: Assessing the role of unconventional policies for a decade of household data
Processing paper: A hybrid stochastic differential reinsurance and investment game with bounded memory
Successfully processed paper: A hybrid stochastic differential reinsurance and investment game with bounded memory
Processing paper: Optimal Investment Under Transaction Costs: A Threshold Rebalanced Portfolio Approach
Successfully processed paper: Optimal Investment Under Transaction Costs: A Threshold Rebalanced Portfolio Approach
Successfully processed 111 document chunks from 3 papers
Loaded 111 document chunks from Arxiv papers.


In [ ]:
### Knowledge Graph Based Synthetic Generation
### Unrolled SDG

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/home/naveen/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/home/naveen/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/home/naveen/code/AIE7/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


In [ ]:
### instantiate our Knowledge Graph.

from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

In [ ]:
### To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/naveen/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/naveen/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [ ]:
### insert each of our full documents into the graph
from ragas.testset.graph import Node, NodeType

for doc in financial_knowledge_resources[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

In [ ]:
### apply the default transformations to our knowledge graph

from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=financial_knowledge_resources, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/18 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/36 [00:00<?, ?it/s]

Property 'summary' already exists in node '612a5b'. Skipping!
Property 'summary' already exists in node '4fa82a'. Skipping!
Property 'summary' already exists in node 'c0ea64'. Skipping!
Property 'summary' already exists in node '5ac7ff'. Skipping!
Property 'summary' already exists in node '6080e1'. Skipping!
Property 'summary' already exists in node '9ff95b'. Skipping!
Property 'summary' already exists in node '776494'. Skipping!
Property 'summary' already exists in node '846395'. Skipping!
Property 'summary' already exists in node '427a8e'. Skipping!
Property 'summary' already exists in node '1e652a'. Skipping!
Property 'summary' already exists in node 'f069ca'. Skipping!
Property 'summary' already exists in node '5da69c'. Skipping!
Property 'summary' already exists in node 'fadbd4'. Skipping!
Property 'summary' already exists in node 'd386ee'. Skipping!
Property 'summary' already exists in node '6bd5ce'. Skipping!
Property 'summary' already exists in node '511c73'. Skipping!
Property

Applying CustomNodeFilter: 0it [00:00, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/36 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '4fa82a'. Skipping!
Property 'summary_embedding' already exists in node '846395'. Skipping!
Property 'summary_embedding' already exists in node 'c0ea64'. Skipping!
Property 'summary_embedding' already exists in node '6080e1'. Skipping!
Property 'summary_embedding' already exists in node '9ff95b'. Skipping!
Property 'summary_embedding' already exists in node '612a5b'. Skipping!
Property 'summary_embedding' already exists in node '1e652a'. Skipping!
Property 'summary_embedding' already exists in node '427a8e'. Skipping!
Property 'summary_embedding' already exists in node '776494'. Skipping!
Property 'summary_embedding' already exists in node '5da69c'. Skipping!
Property 'summary_embedding' already exists in node 'f069ca'. Skipping!
Property 'summary_embedding' already exists in node '5ac7ff'. Skipping!
Property 'summary_embedding' already exists in node 'fadbd4'. Skipping!
Property 'summary_embedding' already exists in node 'd386ee'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 38, relationships: 630)

In [ ]:
### save and load our knowledge graphs 

kg.save("fin_data_kg.json")
fin_data_kg = KnowledgeGraph.load("fin_data_kg.json")
fin_data_kg

KnowledgeGraph(nodes: 38, relationships: 630)

In [ ]:
### Using our knowledge graph, we can construct a "test set generator"
### Abstracted SDG

from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(financial_knowledge_resources[:10], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/9 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/10 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Property 'summary' already exists in node 'c673bd'. Skipping!
Property 'summary' already exists in node 'ac914d'. Skipping!
Property 'summary' already exists in node '0fdcd2'. Skipping!
Property 'summary' already exists in node 'a6e291'. Skipping!
Property 'summary' already exists in node 'a76175'. Skipping!
Property 'summary' already exists in node 'a7ed35'. Skipping!
Property 'summary' already exists in node '162339'. Skipping!
Property 'summary' already exists in node 'b961b4'. Skipping!


...

Property 'summary_embedding' already exists in node '162339'. Skipping!
Property 'summary_embedding' already exists in node 'a7ed35'. Skipping!
Property 'summary_embedding' already exists in node 'a76175'. Skipping!
Property 'summary_embedding' already exists in node '0fdcd2'. Skipping!
Property 'summary_embedding' already exists in node 'c673bd'. Skipping!
Property 'summary_embedding' already exists in node 'ac914d'. Skipping!
Property 'summary_embedding' already exists in node 'a6e291'. Skipping!
Property 'summary_embedding' already exists in node 'b961b4'. Skipping!


Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
### define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

In [ ]:
### we can use our TestSetGenerator to generate our testset!

dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What is the BoE's role in wealth inequality?,[DECEMBER 2019 the savings redistribution chan...,The context discusses the effects of monetary ...,single_hop_specifc_query_synthesizer
1,How does quantitative easing (QE) influence we...,[DECEMBER 2019 the savings redistribution chan...,The analysis indicates that unconventional mon...,single_hop_specifc_query_synthesizer
2,What is the ZLB?,[DECEMBER 2019 the savings redistribution chan...,"The ZLB refers to the zero lower bound, which ...",single_hop_specifc_query_synthesizer
3,Considering the significance of DECEMBER 2019 ...,[DECEMBER 2019 the savings redistribution chan...,The analysis explores the effects of UMP shock...,single_hop_specifc_query_synthesizer
4,How does the zero lower bound (ZLB) influence ...,[DECEMBER 2019 the savings redistribution chan...,The analysis uses a Bayesian threshold VAR to ...,single_hop_specifc_query_synthesizer
5,How do the empirical robustness and sensitivit...,[<1-hop>\n\nDECEMBER 2019 the savings redistri...,The analysis demonstrates that the results reg...,multi_hop_abstract_query_synthesizer
6,"How do the roles of asset prices, financial ma...",[<1-hop>\n\nDECEMBER 2019 the savings redistri...,The context indicates that unconventional mone...,multi_hop_abstract_query_synthesizer
7,How do impulse response analysis and variance ...,[<1-hop>\n\nDECEMBER 2019 the savings redistri...,Impulse response analysis indicates that uncon...,multi_hop_abstract_query_synthesizer
8,How do counterfactual policy analysis and impu...,[<1-hop>\n\nDECEMBER 2019 the savings redistri...,The counterfactual policy analysis allows us t...,multi_hop_abstract_query_synthesizer
9,how counterfactual policy analysis and simulat...,[<1-hop>\n\nDECEMBER 2019 the savings redistri...,the context explains that the study uses count...,multi_hop_abstract_query_synthesizer


In [ ]:
### creating a dataset for LangSmith!

from langsmith import Client

client = Client()

dataset_name = "Financial Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Financial Synthetic Data"
)

In [ ]:
### We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

In [ ]:
### Basic RAG Chain

rag_documents = financial_knowledge_resources

In [ ]:
### use LangChain's recursive character text splitter!

from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

In [ ]:
### create a QDrant VectorStore with the collection name "Financial Queries".
### We'll leverage OpenAI's [`text-embedding-3-small`]

from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    financial_knowledge_resources,
    embeddings,
    location=":memory:",
    collection_name="FinancialQueries"
)

In [ ]:
### Naive RAG Chain
### R - Retrieval

naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

In [ ]:
### A - Augmented
### standard prompt for our simple RAG chain

from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

In [ ]:
### G - Generation
### leverage gpt-4.1-nano as our LLM

from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

In [ ]:
### LCEL RAG Chain

from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model | StrOutputParser()} 
)

In [ ]:
### Let's see how this simple chain does on a few different prompts.

naive_retrieval_chain.invoke({"question" : "What is the most common issue with Financial Analysis?"})["response"]

'Based on the provided context, the most common issue with financial analysis is the challenge of accurately modeling and optimizing investment strategies in markets that involve transaction costs, illiquid assets like housing, and heterogeneous household characteristics. Specifically, incorporating realistic constraints such as transaction costs, illiquidity, and household financial constraints complicates the portfolio optimization process and often leads to underinvestment in certain assets like stocks and deposits. Additionally, aligning theoretical models with actual household behavior remains a persistent challenge.'

In [ ]:
naive_retrieval_chain.invoke({"question" : "What are the three main sources of data for financial analysis?"})["response"]

'The three main sources of data for financial analysis are typically:\n\n1. Financial market data (such as stock prices, interest rates, and asset returns)\n2. Household or consumer financial data (including surveys and questionnaires on asset holdings, income, and expenditure)\n3. Economic and macroeconomic data (such as inflation indices, housing prices, and macroeconomic indicators)\n\nBased on the context provided, especially the use of the Spanish Survey of Household Finance (EFF), the primary sources include financial market data and household financial surveys.'

In [ ]:
naive_retrieval_chain.invoke({"question" : "What are the three basic tools of financial analysis?"})["response"]

"The three basic tools of financial analysis are typically considered to be:\n\n1. **Ratio Analysis** – Examining financial ratios to assess a company's performance, liquidity, profitability, and efficiency.\n2. **Trend Analysis** – Analyzing financial statement data over multiple periods to identify patterns or trends that can indicate future performance.\n3. **Comparative or Cross-Sectional Analysis** – Comparing financial metrics across similar companies or industry benchmarks to evaluate relative performance.\n\nIf you are referring to the context provided, it mainly discusses portfolio choices and investment strategies, but the foundational tools of financial analysis remain these three broad categories."

In [ ]:
### Evaluating the App with Ragas

for test_row in dataset:
  response = naive_retrieval_chain.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

dataset.samples[0].eval_sample.response

"The Bank of England (BoE) plays a significant role in wealth inequality primarily through its monetary policy decisions, especially unconventional measures like asset purchases (quantitative easing). According to the research, these policies can have redistributive effects that influence the distribution of household wealth across different income and wealth groups.\n\nKey points include:\n- Expansionary monetary policies, such as large-scale asset purchases, tend to increase wealth inequality by disproportionately raising the wealth of the richer households who hold more financial and housing assets.\n- Such policies have long-lasting effects, often widening the gap between the wealthy and the less wealthy.\n- The redistribution effects are linked to the revaluation of assets like financial instruments and housing, which are more heavily owned by wealthier households.\n- Evidence suggests that unconventional policies conducted by the BoE, especially during periods like the global fin

In [ ]:
### we can convert that table into a EvaluationDataset

from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

In [ ]:
### select a judge model

from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

In [ ]:
### evaluate on our desired metrics!

from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
result

Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Exception raised in Job[14]: AttributeError('StringIO' object has no attribute 'statements')
Exception raised in Job[17]: AttributeError('StringIO' object has no attribute 'statements')
Exception raised in Job[16]: TimeoutError()
Exception raised in Job[41]: TimeoutError()
Exception raised in Job[47]: TimeoutError()
Exception raised in Job[53]: TimeoutError()


{'context_recall': 0.9667, 'faithfulness': 0.9610, 'factual_correctness': 0.7344, 'answer_relevancy': 0.9444, 'context_entity_recall': 0.2318, 'noise_sensitivity_relevant': 0.4184}

In [ ]:
### LangSmith Evaluation Set-up
###  use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

eval_llm = ChatOpenAI(model="gpt-4.1")

In [ ]:
### using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["response"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

In [ ]:
### LangSmith Evaluation

evaluate(
    naive_retrieval_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "naive_retrieval_chain"},
)

View the evaluation results for experiment: 'ample-powder-73' at:
https://smith.langchain.com/o/d7c4e4d3-71ec-4002-bb25-dc2c0ae939a5/datasets/93331ea9-0298-493e-a04d-ee5bb1d253ae/compare?selectedSessions=516f487b-e6ca-433b-aab3-d2c460a14414




0it [00:00, ?it/s]

,inputs.question,outputs.response,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,how counterfactual policy analysis and simulat...,Counterfactual policy analysis and simulations...,None,the context explains that the study uses count...,1,1,0,9.907944,475fe06b-3690-4dc9-938d-bd54d21c542c,16313db8-f6c9-4ec2-bb29-23c9252e1c93
1,How do counterfactual policy analysis and impu...,Counterfactual policy analysis and impulse res...,None,The counterfactual policy analysis allows us t...,1,1,0,5.982972,372615e1-eb60-48f3-be4d-2cc9af79b7ca,ab833964-d652-4f9a-af09-cdabaac0ba31
2,How do impulse response analysis and variance ...,Impulse response analysis and variance decompo...,None,Impulse response analysis indicates that uncon...,1,1,0,4.791059,9651d4a9-2d5f-4c73-877c-2c7771e26d1c,11030867-9147-42c6-bec4-63a7d713360e
3,"How do the roles of asset prices, financial ma...","The roles of asset prices, financial markets, ...",None,The context indicates that unconventional mone...,1,1,0,13.039078,c0b01aec-5155-4e67-8c57-14e456ddfe34,bbf1ed7c-6a03-4a02-8502-4da6a1c43924
4,How do the empirical robustness and sensitivit...,The empirical robustness and sensitivity analy...,None,The analysis demonstrates that the results reg...,1,1,0,5.238595,c6da17bc-f7ef-4265-a302-8ede71b7f298,4826f378-f5c7-47cf-9eef-2f1969c1372d
5,How does the zero lower bound (ZLB) influence ...,"According to recent analyses, the zero lower b...",None,The analysis uses a Bayesian threshold VAR to ...,1,1,0,4.346933,53ca8d05-a29f-4f73-81c7-c1b23ca864e7,c90077b9-3371-40d9-980f-8e70ae8188d7
6,Considering the significance of DECEMBER 2019 ...,The context indicates that December 2019 is at...,None,The analysis explores the effects of UMP shock...,1,1,0,8.150024,38fe9fe1-b18e-407e-9f8e-6e36cd35048b,06005f52-0cce-4445-8e26-8050a1bb6cee
7,What is the ZLB?,The ZLB stands for the Zero Lower Bound. It re...,None,"The ZLB refers to the zero lower bound, which ...",1,1,0,0.043014,93cda99a-9a01-4f1e-99a0-ea3a5107648f,7bd6e1e6-33f0-4355-8934-d978544ac2f2
8,How does quantitative easing (QE) influence we...,According to the analysis presented in the con...,None,The analysis indicates that unconventional mon...,1,1,0,2.786651,75300a28-0beb-471b-8935-263518fbe43d,16442e61-cf5c-4b10-aa8b-b2436a6e696f
9,What is the BoE's role in wealth inequality?,The Bank of England (BoE) plays a significant ...,None,The context discusses the effects of monetary ...,1,1,0,2.774816,472a692b-6887-4db3-8e42-fd1fe2f6ba5e,f401ffa7-b698-4093-9121-b79522184d44


In [ ]:
### Best-Matching 25 (BM25) Retriever

from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(financial_knowledge_resources, )

In [ ]:
### construct the same chain 

bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model | StrOutputParser() }
)

In [ ]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with Financial Analysis?"})["response"]

'The most common issue with Financial Analysis is the difficulty in disentangling the impacts of different factors and accurately measuring specific effects, such as the impact of unconventional monetary policies on wealth inequality. This is often due to identification problems, overlapping periods of policies (like low interest rates and unconventional policies), and the challenge of isolating the influence of each factor.'

In [ ]:
bm25_retrieval_chain.invoke({"question" : "What are the three main sources of data for financial analysis?"})["response"]

'The three main sources of data for financial analysis are typically:\n\n1. Financial statements and accounting records\n2. Market data (such as stock prices, interest rates, and exchange rates)\n3. Economic and demographic data (such as surveys and government statistics)\n\nIn the specific context provided, the primary data source used for analyzing household finance decisions is the Spanish Survey of Household Finance (EFF), which offers detailed demographic, financial, and economic information about households. Additionally, historical market data, such as stock returns and interest rates from sources like the Bank of Spain and the Statistical Bulletin, are used for financial analysis.\n\nSo, summarizing for your question, the main sources of data are:  \n- Household surveys and demographic/economic data (e.g., the Spanish Survey of Household Finance)  \n- Market data (e.g., stock returns, interest rates)  \n- Historical financial data and statistics from financial institutions and 

In [ ]:
bm25_retrieval_chain.invoke({"question" : "What are the three basic tools of financial analysis?"})["response"]

"The three basic tools of financial analysis are typically considered to be:\n\n1. **Financial Statements Analysis** – Examining the balance sheet, income statement, and cash flow statement to assess a company's financial health.\n\n2. **Ratios and Financial Metrics** – Using ratios such as profitability ratios, liquidity ratios, and leverage ratios to evaluate performance and financial stability.\n\n3. **Trend and Comparative Analysis** – Comparing financial data over time (trend analysis) and against industry peers (comparative analysis) to identify patterns and relative performance.\n\nThese tools help investors and analysts understand the financial condition and prospects of an entity."

In [ ]:
### Evaluating the App with Ragas

for test_row in dataset:
  response = bm25_retrieval_chain.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  

dataset.samples[0].eval_sample.response

"The Bank of England (BoE) plays a significant role in influencing wealth inequality through its monetary policy decisions, particularly unconventional policies like asset purchases and quantitative easing (QE). According to the research, these policies have long-lasting and sometimes redistributive effects on household wealth distribution in Great Britain. \n\nSpecifically, expansionary monetary policies such as asset purchases tend to raise wealth inequality. They do this by increasing asset prices, which benefits wealthier households more, especially those holding financial and housing assets. While some channels, like house price increases, can offset regressive effects, the overall evidence suggests that unconventional monetary policy shocks contribute to widening wealth gaps across households.\n\nFurthermore, the BoE's policies can impact different wealth components (net wealth, housing wealth, financial wealth) and influence wealth distribution via multiple channels, such as the

In [ ]:
### we can convert that table into a EvaluationDataset

from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

In [ ]:
### evaluate on our desired metrics!

from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
result

Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

{'context_recall': 0.9667, 'faithfulness': 0.8398, 'factual_correctness': 0.6330, 'answer_relevancy': 0.9562, 'context_entity_recall': 0.2154, 'noise_sensitivity_relevant': 0.3265}

In [ ]:
### LangSmith Evaluation
from langsmith.evaluation import evaluate
evaluate(
    bm25_retrieval_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "bm25_retrieval_chain"},
)

View the evaluation results for experiment: 'virtual-vacation-19' at:
https://smith.langchain.com/o/d7c4e4d3-71ec-4002-bb25-dc2c0ae939a5/datasets/93331ea9-0298-493e-a04d-ee5bb1d253ae/compare?selectedSessions=e7898e2e-f9d7-48a0-8512-870028fd4c7e




0it [00:00, ?it/s]

,inputs.question,outputs.response,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,how counterfactual policy analysis and simulat...,Counterfactual policy analysis and simulations...,None,the context explains that the study uses count...,1,1,0,8.324132,475fe06b-3690-4dc9-938d-bd54d21c542c,b1cf45f6-7d48-4da0-b09b-554e19340f7f
1,How do counterfactual policy analysis and impu...,Counterfactual policy analysis and impulse res...,None,The counterfactual policy analysis allows us t...,1,1,0,5.843473,372615e1-eb60-48f3-be4d-2cc9af79b7ca,60ddd62f-f36c-4b24-989d-20d0286781d9
2,How do impulse response analysis and variance ...,Impulse response analysis and variance decompo...,None,Impulse response analysis indicates that uncon...,1,1,0,3.567464,9651d4a9-2d5f-4c73-877c-2c7771e26d1c,5e9c9a9b-c7c8-4b81-a6fe-fad258fe0480
3,"How do the roles of asset prices, financial ma...","The roles of asset prices, financial markets, ...",None,The context indicates that unconventional mone...,1,1,0,9.471773,c0b01aec-5155-4e67-8c57-14e456ddfe34,c6bd132f-e6d9-447f-9919-d332736359d6
4,How do the empirical robustness and sensitivit...,The empirical robustness and sensitivity analy...,None,The analysis demonstrates that the results reg...,1,1,0,7.735725,c6da17bc-f7ef-4265-a302-8ede71b7f298,766cb4d4-95d6-4df7-a618-c94ee9d0da99
5,How does the zero lower bound (ZLB) influence ...,"According to recent analyses, the zero lower b...",None,The analysis uses a Bayesian threshold VAR to ...,1,1,0,3.420821,53ca8d05-a29f-4f73-81c7-c1b23ca864e7,fdb0796a-d937-441e-80bc-f7095e725269
6,Considering the significance of DECEMBER 2019 ...,Considering December 2019 within the context o...,None,The analysis explores the effects of UMP shock...,1,1,0,5.583098,38fe9fe1-b18e-407e-9f8e-6e36cd35048b,dfc1821f-2802-4d00-ac8d-f8bb85d9c929
7,What is the ZLB?,"The ZLB refers to the ""Zero Lower Bound,"" whic...",None,"The ZLB refers to the zero lower bound, which ...",0,1,0,0.811520,93cda99a-9a01-4f1e-99a0-ea3a5107648f,2c3fd8b3-7c06-4b54-b4b5-b542bac065a4
8,How does quantitative easing (QE) influence we...,According to the analysis presented in the con...,None,The analysis indicates that unconventional mon...,1,1,0,1.485050,75300a28-0beb-471b-8935-263518fbe43d,435bea2d-7a22-498a-9a76-36bb7049031e
9,What is the BoE's role in wealth inequality?,The Bank of England's (BoE) role in wealth ine...,None,The context discusses the effects of monetary ...,1,1,0,3.082091,472a692b-6887-4db3-8e42-fd1fe2f6ba5e,c6aa0477-1b97-4d42-88b8-1e240cc023b1


In [ ]:
### Contextual Compression (Using Reranking)
### Load Cohere API Key

os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

In [ ]:
### create our chain

contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model | StrOutputParser()}
)

In [ ]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with Financial Analysis?"})["response"]

"The most common issue with financial analysis, based on the context provided, is that households tend to significantly underinvest in stocks and deposits relative to the optimal portfolio, often due to financial constraints, asset illiquidity, or risk considerations. Additionally, a recurring challenge is managing assets that are indivisible and illiquid, such as housing, which restricts portfolio choices and complicates investment decisions. \n\nIn a broader sense, common issues include the misallocation of assets, constraints faced by investors (like financial constraints and transaction costs), and the difficulty in achieving optimal investment strategies, especially when dealing with illiquid assets like housing.  \n\nIf you were referring to a specific flaw or challenge in financial analysis in general, the literature suggests that issues like data limitations, modeling assumptions, and behavioral biases are also prevalent. However, based on the context you provided, the key issu

In [ ]:
contextual_compression_retrieval_chain.invoke({"question" : "What are the three main sources of data for financial analysis?"})["response"]

'The three main sources of data for financial analysis are:\n\n1. Market data, such as asset returns and interest rates (e.g., stock market returns, bank interest rates, housing price indices).\n2. Household or individual financial data, such as survey data on household assets, debts, and demographics (e.g., the Spanish Survey of Household Finance used in the study).\n3. External economic and financial indicators, such as housing price indices, property taxes, and macroeconomic variables from official sources.\n\nIf you need further clarification or details, feel free to ask!'

In [ ]:
contextual_compression_retrieval_chain.invoke({"question" : "What are the three basic tools of financial analysis?"})["response"]

'The three basic tools of financial analysis are typically considered to be the income statement, the balance sheet, and the cash flow statement.'

In [ ]:
### Evaluating the App with Ragas

for test_row in dataset:
  response = contextual_compression_retrieval_chain.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  

dataset.samples[0].eval_sample.response



"The Bank of England's (BoE) role in wealth inequality involves its monetary policy decisions, particularly during and after the Global Financial Crisis. According to the research, unconventional monetary policies such as asset purchases (quantitative easing) implemented by the BoE have significant and long-lasting redistributive effects on household wealth. Specifically, expansionary monetary policies in the form of asset purchases tend to increase wealth inequality across households, as measured by Gini coefficients of net wealth, housing wealth, and financial wealth. \n\nThis means that the BoE's policies aimed at stimulating the economy through unconventional measures can inadvertently favor wealthier households, thereby widening the gap between different socioeconomic groups. The evidence suggests that central banks, including the BoE, need to be aware of the potential for their policies to influence wealth disparities, which can have broader implications for economic inequality a

In [ ]:
### we can convert that table into a EvaluationDataset

from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())



In [ ]:
### evaluate on our desired metrics!

from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
result


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

{'context_recall': 1.0000, 'faithfulness': 0.9237, 'factual_correctness': 0.6050, 'answer_relevancy': 0.9507, 'context_entity_recall': 0.2083, 'noise_sensitivity_relevant': 0.2855}

In [ ]:
### LangSmith Evaluation
from langsmith.evaluation import evaluate
evaluate(
    contextual_compression_retrieval_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "contextual_compression_retrieval_chain"},
)

View the evaluation results for experiment: 'unique-seed-90' at:
https://smith.langchain.com/o/d7c4e4d3-71ec-4002-bb25-dc2c0ae939a5/datasets/93331ea9-0298-493e-a04d-ee5bb1d253ae/compare?selectedSessions=c87c62e2-36a9-4a35-8756-5a3c3ccf0007




0it [00:00, ?it/s]

,inputs.question,outputs.response,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,how counterfactual policy analysis and simulat...,Counterfactual policy analysis and simulations...,None,the context explains that the study uses count...,1,1,0,7.494388,475fe06b-3690-4dc9-938d-bd54d21c542c,2370046b-d882-444f-ac2d-054f1aff2bca
1,How do counterfactual policy analysis and impu...,Counterfactual policy analysis and impulse res...,None,The counterfactual policy analysis allows us t...,1,1,0,5.216393,372615e1-eb60-48f3-be4d-2cc9af79b7ca,5bfaeddb-18f0-4457-90f3-8bc193d82d59
2,How do impulse response analysis and variance ...,Impulse response analysis and variance decompo...,None,Impulse response analysis indicates that uncon...,1,1,0,3.577714,9651d4a9-2d5f-4c73-877c-2c7771e26d1c,d9ce8d59-6ff3-4eee-840e-94d9389491b3
3,"How do the roles of asset prices, financial ma...","The roles of asset prices, financial markets, ...",None,The context indicates that unconventional mone...,1,1,0,11.030203,c0b01aec-5155-4e67-8c57-14e456ddfe34,49525204-8ef9-4f4a-b52a-a21237d5d585
4,How do the empirical robustness and sensitivit...,The empirical robustness and sensitivity analy...,None,The analysis demonstrates that the results reg...,1,1,0,0.918665,c6da17bc-f7ef-4265-a302-8ede71b7f298,bd55e8bd-d533-490e-8695-aea118e10f81
5,How does the zero lower bound (ZLB) influence ...,"According to recent analyses, the zero lower b...",None,The analysis uses a Bayesian threshold VAR to ...,1,1,0,6.662487,53ca8d05-a29f-4f73-81c7-c1b23ca864e7,263d1530-dffa-44f8-acd2-3ea4651e91da
6,Considering the significance of DECEMBER 2019 ...,The unconventional monetary policy shocks duri...,None,The analysis explores the effects of UMP shock...,1,1,0,6.679382,38fe9fe1-b18e-407e-9f8e-6e36cd35048b,fec43d58-6699-4e46-9679-a9a44fb38c4b
7,What is the ZLB?,"The ZLB refers to the Zero Lower Bound, which ...",None,"The ZLB refers to the zero lower bound, which ...",1,1,0,1.998433,93cda99a-9a01-4f1e-99a0-ea3a5107648f,72cb13e1-06e6-4639-bb73-da9129aca1ab
8,How does quantitative easing (QE) influence we...,According to the analysis presented in the con...,None,The analysis indicates that unconventional mon...,1,1,0,3.887352,75300a28-0beb-471b-8935-263518fbe43d,95467e9f-68cc-484b-a9cf-ce679b65594a
9,What is the BoE's role in wealth inequality?,The Bank of England's (BoE) role in wealth ine...,None,The context discusses the effects of monetary ...,1,1,0,3.048947,472a692b-6887-4db3-8e42-fd1fe2f6ba5e,edb83bbc-f067-4430-a0b9-f121649f34b7


In [ ]:
### Multi-Query Retriever

from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [ ]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model | StrOutputParser()}
)

In [ ]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with Financial Analysis?"})["response"]

'The most common issue with financial analysis, as suggested by the context, relates to the challenges in optimizing portfolio choices and investment strategies under real-world constraints such as transaction costs, illiquid assets, financial constraints, and market risks. Specifically, the literature and studies mentioned highlight difficulties in accurately modeling and maximizing wealth due to factors like transaction costs, incomplete markets, illiquid assets (like housing), and constraints faced by households. \n\nIn summary, a key issue is **the complexity of accurately modeling and optimizing investment portfolios in realistic settings**, which often leads to underinvestment or suboptimal allocation due to market frictions, behavioral biases, or incomplete information.\n\nIf you were looking for a more specific or different answer, I do not have a direct statement from the provided text explicitly pointing to a single most common issue but based on the themes, the prevalent cha

In [ ]:
multi_query_retrieval_chain.invoke({"question" : "What are the three main sources of data for financial analysis?"})["response"]

"The three main sources of data for financial analysis, as indicated in the context, are:\n\n1. Households' Demographic Characteristics – including age, education level, employment sector, and income.\n2. Financial Assets Data – such as holdings in stocks, bank time deposits, mortgages, and housing, obtained from surveys like the Spanish Survey of Household Finance (EFF).\n3. Historical Asset Return Data – including annual real returns of financial assets like stocks, deposits, mortgages, and housing prices over time, often used to analyze investment performance and risk.\n\nLet me know if you'd like more details or further clarification!"

In [ ]:
multi_query_retrieval_chain.invoke({"question" : "What are the three basic tools of financial analysis?"})["response"]

'The three basic tools of financial analysis are generally considered to be:\n\n1. **Financial Ratios** – Measures like profitability, liquidity, and solvency ratios that help evaluate a company’s financial health.\n2. **Financial Statements Analysis** – Examining the balance sheet, income statement, and cash flow statement to assess performance.\n3. **Financial Forecasting and Planning** – Projecting future earnings, cash flows, and financial needs to inform investment and management decisions.\n\nIf you are referring specifically to concepts from the provided context, the document discusses portfolio choice and investment decisions, but it does not explicitly list the "three basic tools of financial analysis." Based on standard financial analysis principles, the above tools are considered fundamental.'

In [ ]:
### Evaluating the App with Ragas

for test_row in dataset:
  response = multi_query_retrieval_chain.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  

dataset.samples[0].eval_sample.response



"The Bank of England (BoE) plays an influential role in wealth inequality primarily through its monetary policy decisions, especially during unconventional measures like quantitative easing (QE). According to the research, unconventional monetary policy shocks—such as asset purchases—have long-lasting redistributive effects, generally increasing wealth inequality across households.\n\nSpecifically, the BoE's asset purchase programs tend to raise asset prices, benefiting wealthier households who hold more financial assets and property. This increases their wealth and widens the income and wealth gap between the rich and the rest of the population. The studies show that expansionary monetary policies in the form of asset purchases can lead to a rise in the Gini coefficient, indicating higher wealth inequality.\n\nFurthermore, these policies also influence housing prices, which can benefit households with property ownership but may also have complex effects depending on ownership patterns

In [ ]:
### we can convert that table into a EvaluationDataset

from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())


In [ ]:
### evaluate on our desired metrics!

from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
result


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Exception raised in Job[13]: AttributeError('StringIO' object has no attribute 'statements')
Exception raised in Job[1]: AttributeError('StringIO' object has no attribute 'statements')


{'context_recall': 1.0000, 'faithfulness': 0.9384, 'factual_correctness': 0.6660, 'answer_relevancy': 0.9467, 'context_entity_recall': 0.2044, 'noise_sensitivity_relevant': 0.3196}

In [ ]:
### LangSmith Evaluation
from langsmith.evaluation import evaluate
evaluate(
    multi_query_retrieval_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "multi_query_retrieval_chain"},
)


View the evaluation results for experiment: 'healthy-operation-78' at:
https://smith.langchain.com/o/d7c4e4d3-71ec-4002-bb25-dc2c0ae939a5/datasets/93331ea9-0298-493e-a04d-ee5bb1d253ae/compare?selectedSessions=8ad94438-ff71-4e22-b121-582b94006dee




0it [00:00, ?it/s]

,inputs.question,outputs.response,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,how counterfactual policy analysis and simulat...,Counterfactual policy analysis and simulations...,None,the context explains that the study uses count...,1,1,0,11.124853,475fe06b-3690-4dc9-938d-bd54d21c542c,eb7054da-2cd8-4d11-b9cb-653afc9347ea
1,How do counterfactual policy analysis and impu...,Counterfactual policy analysis and impulse res...,None,The counterfactual policy analysis allows us t...,1,1,0,9.246240,372615e1-eb60-48f3-be4d-2cc9af79b7ca,ab15156e-80de-4ead-859f-a03666679f88
2,How do impulse response analysis and variance ...,Impulse response analysis and variance decompo...,None,Impulse response analysis indicates that uncon...,1,1,0,7.607245,9651d4a9-2d5f-4c73-877c-2c7771e26d1c,10f4ffaf-a644-46c3-ad49-b639ef73f35c
3,"How do the roles of asset prices, financial ma...","The roles of asset prices, financial markets, ...",None,The context indicates that unconventional mone...,1,1,0,17.062581,c0b01aec-5155-4e67-8c57-14e456ddfe34,c7879b64-cfdc-4bd2-9e57-bf6e8c25b7dd
4,How do the empirical robustness and sensitivit...,The empirical robustness and sensitivity analy...,None,The analysis demonstrates that the results reg...,1,1,0,9.847886,c6da17bc-f7ef-4265-a302-8ede71b7f298,e535a27f-44b6-42cd-9284-64c076e5ecc0
5,How does the zero lower bound (ZLB) influence ...,According to recent analyses presented in the ...,None,The analysis uses a Bayesian threshold VAR to ...,1,1,0,7.690753,53ca8d05-a29f-4f73-81c7-c1b23ca864e7,2e2abb88-f59f-426d-8fc7-cead2b17c5f5
6,Considering the significance of DECEMBER 2019 ...,"Based on the provided research and analysis, D...",None,The analysis explores the effects of UMP shock...,1,1,0,8.922711,38fe9fe1-b18e-407e-9f8e-6e36cd35048b,d40b4106-f295-49f2-b338-324484d865fd
7,What is the ZLB?,"The ZLB, or Zero Lower Bound, refers to the si...",None,"The ZLB refers to the zero lower bound, which ...",1,1,0,2.132182,93cda99a-9a01-4f1e-99a0-ea3a5107648f,9eafaa20-923e-482f-9e3c-6e54dcc91c5e
8,How does quantitative easing (QE) influence we...,According to the analysis presented in the con...,None,The analysis indicates that unconventional mon...,1,1,0,6.168513,75300a28-0beb-471b-8935-263518fbe43d,2d52af38-9b47-4596-92fc-b53705774b1b
9,What is the BoE's role in wealth inequality?,The Bank of England's (BoE) plays a significan...,None,The context discusses the effects of monetary ...,1,1,0,8.750550,472a692b-6887-4db3-8e42-fd1fe2f6ba5e,ee1b535f-3aba-4d1c-af21-fa7f87a69346


In [ ]:
### Parent Document Retriever

from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = financial_knowledge_resources
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

In [ ]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

In [ ]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

In [ ]:
parent_document_retriever.add_documents(parent_docs, ids=None)

In [ ]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model| StrOutputParser()}
)

In [ ]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with Financial Analysis?"})["response"]

'The most common issue with financial analysis, as generally acknowledged, is accurately accounting for market complexities such as transaction costs, market volatility, and data limitations. Specifically, in the context of the provided documents, a key challenge highlighted is managing proportional transaction costs in portfolio management and accurately modeling their impact on investment outcomes.'

In [ ]:
parent_document_retrieval_chain.invoke({"question" : "What are the three main sources of data for financial analysis?"})["response"]

'The three main sources of data for financial analysis are:\n\n1. Market Data (such as stock prices, interest rates, and asset prices)\n2. Financial Statements and Reports (including balance sheets, income statements, and cash flow statements)\n3. Economic and Demographic Data (like macroeconomic indicators, employment figures, and household financial information)\n\nIf you need more specific details or examples, feel free to ask!'

In [ ]:
parent_document_retrieval_chain.invoke({"question" : "What are the three basic tools of financial analysis?"})["response"]

"The three basic tools of financial analysis are typically considered to be:\n\n1. **Ratio Analysis** – evaluating financial statements through ratios to assess profitability, liquidity, efficiency, and solvency.\n2. **Financial Statements Analysis** – examining income statements, balance sheets, and cash flow statements to understand a company's financial health.\n3. **Cash Flow Analysis** – analyzing inflows and outflows of cash to evaluate liquidity and financial stability.\n\nPlease note that the provided context does not explicitly list these tools, but based on standard financial analysis principles, these are recognized as the three basic tools."

In [ ]:
### Evaluating the App with Ragas

for test_row in dataset:
  response = parent_document_retrieval_chain.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  

dataset.samples[0].eval_sample.response

"The Bank of England's (BoE) role in wealth inequality primarily involves the implementation of monetary policies that can have redistributive effects on household wealth. According to the research, unconventional monetary policy measures, such as asset purchases and quantitative easing (QE), have significant long-lasting impacts on wealth inequality. Specifically, expansionary monetary policies in the form of asset purchases tend to increase wealth inequality across households by raising the wealth of the richer segments, as measured by indicators like the Gini coefficients of net, housing, and financial wealth. \n\nThe BoE's policies influence asset prices and household portfolios through various channels—such as revaluating assets, affecting housing market prices, and altering borrowing and savings dynamics—which can exacerbate disparities between different household groups. Recognizing this, the BoE's monetary policy decisions are important not only for macroeconomic stability but 

In [ ]:
### we can convert that table into a EvaluationDataset

from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())



In [ ]:
### evaluate on our desired metrics!

from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
result


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

{'context_recall': 1.0000, 'faithfulness': 0.8233, 'factual_correctness': 0.6760, 'answer_relevancy': 0.9559, 'context_entity_recall': 0.2130, 'noise_sensitivity_relevant': 0.2397}

In [ ]:
### LangSmith Evaluation
from langsmith.evaluation import evaluate
evaluate(
    parent_document_retrieval_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "parent_document_retrieval_chain"},
)

View the evaluation results for experiment: 'enchanted-plane-97' at:
https://smith.langchain.com/o/d7c4e4d3-71ec-4002-bb25-dc2c0ae939a5/datasets/93331ea9-0298-493e-a04d-ee5bb1d253ae/compare?selectedSessions=1e757872-e375-48c9-af7e-3578ca950545




0it [00:00, ?it/s]

,inputs.question,outputs.response,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,how counterfactual policy analysis and simulat...,Counterfactual policy analysis and simulations...,None,the context explains that the study uses count...,1,1,0,11.123683,475fe06b-3690-4dc9-938d-bd54d21c542c,39a36360-f908-4865-85dc-1a048eb78622
1,How do counterfactual policy analysis and impu...,Counterfactual policy analysis and impulse res...,None,The counterfactual policy analysis allows us t...,1,1,0,7.966225,372615e1-eb60-48f3-be4d-2cc9af79b7ca,775c198d-dc51-4d83-97e2-69859fbd0f15
2,How do impulse response analysis and variance ...,Impulse response analysis and variance decompo...,None,Impulse response analysis indicates that uncon...,1,1,0,3.676052,9651d4a9-2d5f-4c73-877c-2c7771e26d1c,4608e64d-2358-4bf9-a776-60bfec6ae4df
3,"How do the roles of asset prices, financial ma...","The roles of asset prices, financial markets, ...",None,The context indicates that unconventional mone...,1,1,0,15.417734,c0b01aec-5155-4e67-8c57-14e456ddfe34,56fb20fb-0d33-4405-a28b-d19c8dc70dab
4,How do the empirical robustness and sensitivit...,The empirical robustness and sensitivity analy...,None,The analysis demonstrates that the results reg...,1,1,0,2.049849,c6da17bc-f7ef-4265-a302-8ede71b7f298,22f85378-8350-4ac0-9c5d-ce1d711e95cf
5,How does the zero lower bound (ZLB) influence ...,"According to recent analyses, the zero lower b...",None,The analysis uses a Bayesian threshold VAR to ...,1,1,0,5.466640,53ca8d05-a29f-4f73-81c7-c1b23ca864e7,7fd1d7e1-b3cf-4f93-8584-cdbc80ca11c8
6,Considering the significance of DECEMBER 2019 ...,The unconventional monetary policy shocks arou...,None,The analysis explores the effects of UMP shock...,1,1,0,5.927764,38fe9fe1-b18e-407e-9f8e-6e36cd35048b,7d507736-58c9-4e71-b330-c1d7d899e4c1
7,What is the ZLB?,"The ZLB refers to the Zero Lower Bound, which ...",None,"The ZLB refers to the zero lower bound, which ...",1,1,0,1.321081,93cda99a-9a01-4f1e-99a0-ea3a5107648f,94b5110a-8478-4744-a9df-b7fa5b9ca10c
8,How does quantitative easing (QE) influence we...,According to the analysis presented in the con...,None,The analysis indicates that unconventional mon...,1,1,0,2.890330,75300a28-0beb-471b-8935-263518fbe43d,26b63cbb-e886-4986-b85b-d70c8c524ae4
9,What is the BoE's role in wealth inequality?,The Bank of England's (BoE) plays a significan...,None,The context discusses the effects of monetary ...,1,1,0,3.392933,472a692b-6887-4db3-8e42-fd1fe2f6ba5e,c9626041-86a1-4021-b818-96e5e40b88db


In [ ]:
### Ensemble Retriever

from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

In [ ]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model| StrOutputParser()}
)

In [ ]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with Financial Analysis?"})["response"]

'Based on the provided context, it appears that a common issue with financial analysis is the underinvestment or misallocation of assets by households or investors. Several studies cited indicate that households tend to significantly underinvest in stocks and deposits compared to their optimal portfolios, often due to financial constraints, lack of financial sophistication, or behavioral biases. Additionally, issues such as financial constraints, illiquidity of assets like housing, and the challenge of accurately estimating or adjusting portfolios under transaction costs are recurrent themes.\n\nTherefore, the most common issue with financial analysis is:\n\n**The tendency of investors and households to underinvest in certain asset classes, such as stocks and deposits, often driven by financial constraints, lack of financial knowledge, or market frictions, leading to suboptimal portfolio allocations.**'

In [ ]:
ensemble_retrieval_chain.invoke({"question" : "What are the three main sources of data for financial analysis?"})["response"]

"The three main sources of data for financial analysis are:\n\n1. Household financial surveys or datasets, such as the Spanish Survey of Household Finance (EFF), which provide detailed information on household demographics, wealth, assets, and financial decisions.\n\n2. Historical financial returns data, including data on asset returns like stocks, bonds, deposits, and housing, often obtained from financial institutions, government agencies, or market indices.\n\n3. Macroeconomic and financial market data, such as interest rates, housing price indices, and macroeconomic indicators, generally sourced from official statistics agencies or central banks.\n\nIf you'd like, I can provide more details or context regarding these sources."

In [ ]:
ensemble_retrieval_chain.invoke({"question" : "What are the three basic tools of financial analysis?"})["response"]

"The three basic tools of financial analysis are typically considered to be:\n\n1. **Horizontal Analysis:** Comparing financial data over different periods to identify trends and growth patterns.\n\n2. **Vertical Analysis:** Analyzing financial statements by expressing each item as a percentage of a base figure, such as sales or total assets.\n\n3. **Ratio Analysis:** Using financial ratios to evaluate the company's performance and financial health, such as liquidity ratios, profitability ratios, and solvency ratios.\n\nIf you have any further questions or need more details, feel free to ask!"

In [ ]:
### Evaluating the App with Ragas

for test_row in dataset:
  response = ensemble_retrieval_chain.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  
dataset.samples[0].eval_sample.response

'The Bank of England (BoE) influences wealth inequality primarily through its monetary policy decisions, especially unconventional policies like asset purchases and quantitative easing (QE). According to the research, expansionary monetary policies—such as asset purchases—have been shown to increase wealth inequality over the long term. Specifically, these policies tend to raise the distribution of wealth among households, disproportionately benefiting wealthier households who hold more financial and housing assets. \n\nThe mechanisms through which the BoE’s policies impact wealth inequality include:\n- **Asset Revaluation:** QE and asset purchases boost asset prices, which tend to benefit households holding these assets, typically the wealthier segments.\n- **Housing Market Effects:** UMP measures can inflate house prices, potentially increasing housing wealth for homeowners, who are generally wealthier.\n- **Portfolio Rebalancing:** Lower yields on bonds and other financial assets le

In [ ]:
### we can convert that table into a EvaluationDataset

from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())


In [ ]:
### evaluate on our desired metrics!

from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
result

Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

{'context_recall': 1.0000, 'faithfulness': 0.9633, 'factual_correctness': 0.6280, 'answer_relevancy': 0.9510, 'context_entity_recall': 0.1968, 'noise_sensitivity_relevant': 0.3558}

In [ ]:
### LangSmith Evaluation
from langsmith.evaluation import evaluate
evaluate(
    ensemble_retrieval_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "ensemble_retrieval_chain"},
)

View the evaluation results for experiment: 'spotless-card-64' at:
https://smith.langchain.com/o/d7c4e4d3-71ec-4002-bb25-dc2c0ae939a5/datasets/93331ea9-0298-493e-a04d-ee5bb1d253ae/compare?selectedSessions=48ec74fd-d60c-4ebe-82a9-044912351566




0it [00:00, ?it/s]

,inputs.question,outputs.response,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,how counterfactual policy analysis and simulat...,Counterfactual policy analysis and simulations...,None,the context explains that the study uses count...,1,1,0,12.389448,475fe06b-3690-4dc9-938d-bd54d21c542c,0b90535c-9c18-4497-9b25-fe6f798e37d7
1,How do counterfactual policy analysis and impu...,Counterfactual policy analysis and impulse res...,None,The counterfactual policy analysis allows us t...,1,1,0,12.310721,372615e1-eb60-48f3-be4d-2cc9af79b7ca,ffa3d417-c0a3-44d5-ae4e-eb81251c7fd5
2,How do impulse response analysis and variance ...,Impulse response analysis and variance decompo...,None,Impulse response analysis indicates that uncon...,1,1,0,8.973937,9651d4a9-2d5f-4c73-877c-2c7771e26d1c,44d90905-654c-4749-8b3a-610618259c02
3,"How do the roles of asset prices, financial ma...","The roles of asset prices, financial markets, ...",None,The context indicates that unconventional mone...,1,1,0,14.724517,c0b01aec-5155-4e67-8c57-14e456ddfe34,824e5b39-8a8b-4c1e-8722-58169b6203a6
4,How do the empirical robustness and sensitivit...,The empirical robustness and sensitivity analy...,None,The analysis demonstrates that the results reg...,1,1,0,11.561584,c6da17bc-f7ef-4265-a302-8ede71b7f298,da809e79-4b84-4531-93ee-2d3421272565
5,How does the zero lower bound (ZLB) influence ...,"According to recent analyses, the zero lower b...",None,The analysis uses a Bayesian threshold VAR to ...,1,1,0,7.190591,53ca8d05-a29f-4f73-81c7-c1b23ca864e7,6bc9dcc8-ab9a-458f-8b1b-8ba1f32da328
6,Considering the significance of DECEMBER 2019 ...,"Considering the context of December 2019, the ...",None,The analysis explores the effects of UMP shock...,1,1,0,11.029034,38fe9fe1-b18e-407e-9f8e-6e36cd35048b,7af8a595-bb16-4426-a3aa-eb045c737dde
7,What is the ZLB?,"The ZLB refers to the Zero Lower Bound, which ...",None,"The ZLB refers to the zero lower bound, which ...",1,1,0,3.169072,93cda99a-9a01-4f1e-99a0-ea3a5107648f,dd2c20b1-7b2f-4884-903a-a1f4ea3f5a04
8,How does quantitative easing (QE) influence we...,According to the analysis presented in the con...,None,The analysis indicates that unconventional mon...,1,1,0,8.075756,75300a28-0beb-471b-8935-263518fbe43d,f2aaad8c-e127-4371-b05d-c24f50aad511
9,What is the BoE's role in wealth inequality?,The Bank of England (BoE) influences wealth in...,None,The context discusses the effects of monetary ...,1,1,0,7.186110,472a692b-6887-4db3-8e42-fd1fe2f6ba5e,c96a56c8-034e-4696-97a6-a25df17e882d


In [ ]:
### Semantic Chunking

from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

In [ ]:
semantic_documents = semantic_chunker.split_documents(financial_knowledge_resources[:20])

In [ ]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="financial_knowledge_resources_Semantic_Chunks"
)

In [ ]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

In [ ]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model| StrOutputParser()}
)

In [ ]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with Financial Analysis?"})["response"]

'The most common issue with Financial Analysis, as reflected in the provided context, is the challenge of accurately capturing and analyzing the multiple distributional dimensions of wealth and monetary policy effects, especially due to data limitations. Specifically, the difficulty in measuring wealth inequality accurately arises from insufficient or biased data, such as under-representation of the wealthiest households, and the complexity of disentangling the effects of different policy measures on various components of wealth. These issues make it hard to assess the true impact of policies and the overall distributional outcomes reliably.'

In [ ]:
semantic_retrieval_chain.invoke({"question" : "What are the three main sources of data for financial analysis?"})["response"]

"The three main sources of data for financial analysis are:\n\n1. Household Balance Sheet Data: Provides detailed information on household assets, debts, and wealth components. An example from the context is the Wealth and Asset Survey, which offers monthly and annual data on household assets, liabilities, and wealth distribution.\n\n2. Central Bank and Monetary Policy Indicators: Includes data such as the central bank's balance sheets, total assets, shadow rates, and measures of unconventional monetary policy (UMP) impacts. These proxies are used to assess the stance of monetary policy and its effects on financial markets.\n\n3. Microeconomic and Survey Data on Asset Ownership: Encompasses data on the distribution of different asset types held by households, such as financial assets, housing assets, and liabilities. For instance, the survey data used to construct wealth inequality indices across various percentiles.\n\nThese data sources collectively enable comprehensive analysis of f

In [ ]:
semantic_retrieval_chain.invoke({"question" : "What are the three basic tools of financial analysis?"})["response"]

"The three basic tools of financial analysis are:\n\n1. **Financial Statements Analysis**: Examining balance sheets, income statements, and cash flow statements to assess a company's financial health.\n2. **Ratio Analysis**: Using financial ratios such as liquidity ratios, profitability ratios, and leverage ratios to evaluate performance and financial stability.\n3. **Trend Analysis**: Analyzing financial data over time to identify patterns, growth, or decline in financial metrics.\n\nPlease let me know if you'd like more details on these tools!"

In [ ]:
### Evaluating the App with Ragas

for test_row in dataset:
  response = semantic_retrieval_chain.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  

dataset.samples[0].eval_sample.response


"The Bank of England (BoE) plays a significant role in influencing wealth inequality through its monetary policy decisions, particularly unconventional policies such as quantitative easing (QE). Based on the provided context, the BoE's use of such policies has been shown to have long-lasting redistributive effects on household wealth in Great Britain.\n\nSpecifically, expansionary monetary policies like asset purchases tend to raise overall wealth inequality. The evidence from the study indicates that these policies increase measures of household wealth concentration, such as the Gini coefficient of net wealth, housing wealth, and financial wealth, meaning that the wealthiest households tend to benefit more from these policies. This occurs through mechanisms like revaluing assets, rising asset prices, and portfolio rebalancing, which disproportionately advantage those with larger asset holdings—typically wealthier households.\n\nMoreover, the BoE's policies impact different components 

In [ ]:
### we can convert that table into a EvaluationDataset

from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())


In [ ]:
### evaluate on our desired metrics!

from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
result


Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

Exception raised in Job[5]: AttributeError('StringIO' object has no attribute 'statements')


{'context_recall': 1.0000, 'faithfulness': 0.8695, 'factual_correctness': 0.6480, 'answer_relevancy': 0.9565, 'context_entity_recall': 0.2034, 'noise_sensitivity_relevant': 0.2869}

In [ ]:
### LangSmith Evaluation
from langsmith.evaluation import evaluate
evaluate(
    semantic_retrieval_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "semantic_retrieval_chain"},
)


View the evaluation results for experiment: 'large-advertising-1' at:
https://smith.langchain.com/o/d7c4e4d3-71ec-4002-bb25-dc2c0ae939a5/datasets/93331ea9-0298-493e-a04d-ee5bb1d253ae/compare?selectedSessions=df4c9067-8ed4-4519-bcc2-6d04a86d3e96




0it [00:00, ?it/s]

,inputs.question,outputs.response,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,how counterfactual policy analysis and simulat...,Counterfactual policy analysis and simulations...,None,the context explains that the study uses count...,1,1,0,4.445606,475fe06b-3690-4dc9-938d-bd54d21c542c,4147f07c-dc43-45d0-913d-8eec3a4d57fd
1,How do counterfactual policy analysis and impu...,Counterfactual policy analysis and impulse res...,None,The counterfactual policy analysis allows us t...,1,1,0,5.253246,372615e1-eb60-48f3-be4d-2cc9af79b7ca,9aa47561-c34d-4578-a2a5-de552b9229e6
2,How do impulse response analysis and variance ...,Impulse response analysis and variance decompo...,None,Impulse response analysis indicates that uncon...,1,1,0,3.365615,9651d4a9-2d5f-4c73-877c-2c7771e26d1c,04445182-36e2-4b41-a23a-0c611d0b42fb
3,"How do the roles of asset prices, financial ma...","The roles of asset prices, financial markets, ...",None,The context indicates that unconventional mone...,1,1,0,12.636500,c0b01aec-5155-4e67-8c57-14e456ddfe34,64cb8a76-4f0a-4513-8c71-e27956e6fa8f
4,How do the empirical robustness and sensitivit...,The empirical robustness and sensitivity analy...,None,The analysis demonstrates that the results reg...,1,1,0,2.721663,c6da17bc-f7ef-4265-a302-8ede71b7f298,ce36671e-cd50-476f-b857-7e98fa585f0c
5,How does the zero lower bound (ZLB) influence ...,"According to recent analyses, the zero lower b...",None,The analysis uses a Bayesian threshold VAR to ...,1,1,0,5.223240,53ca8d05-a29f-4f73-81c7-c1b23ca864e7,cc1dd025-6b19-4e1d-9bcb-1332f241e8e7
6,Considering the significance of DECEMBER 2019 ...,"Based on the provided context, December 2019 m...",None,The analysis explores the effects of UMP shock...,1,1,0,6.483469,38fe9fe1-b18e-407e-9f8e-6e36cd35048b,96a95762-58ae-4aa6-bf9f-99d6252c1881
7,What is the ZLB?,The Zero Lower Bound (ZLB) refers to a situati...,None,"The ZLB refers to the zero lower bound, which ...",1,1,0,2.490600,93cda99a-9a01-4f1e-99a0-ea3a5107648f,749577cf-25d4-499b-9d11-339ce5aea663
8,How does quantitative easing (QE) influence we...,According to the analysis presented in the con...,None,The analysis indicates that unconventional mon...,1,1,0,1.919472,75300a28-0beb-471b-8935-263518fbe43d,802dce96-9662-421d-a043-f59dcac85ba7
9,What is the BoE's role in wealth inequality?,The Bank of England (BoE) plays a significant ...,None,The context discusses the effects of monetary ...,1,1,0,2.397031,472a692b-6887-4db3-8e42-fd1fe2f6ba5e,28f4dcab-afee-4f4d-b1cf-9b81488f7278


Retrivers Ragas Metrics:

| Retriver | context_recall | faithfulness |factual_correctness | answer_relevancy | context_entity_recall |noise_sensitivity_relevant
|-----------------|-----------------|-----------------|-----------------|-----------------|-----------------|-----------------|
| naive_retrieval    | 0.9667   | 0.9610    | 0.7344    |0.9444    |0.2318    |0.4184   |
| bm25_retriever    | 0.9667    | 0.8398    | 0.6330    |0.9562    |0.2154    |0.3265    |
| compression_retriever   | 1.0000    | 0.9237    | 0.6050    |0.9507    |0.2083    |0.2855    |
| multi_query_retriever    | 1.0000   | 0.9384    | 0.6660   |0.9467    |0.2044    |0.3196    |
| parent_document_retriever    | 1.0000    | 0.8233    | 0.6760    |0.9559    |0.2130    |0.2397    |
| ensemble_retriever    | 1.0000    | 0.9633    | 0.6280    |0.9510    |0.1968    |0.3558    |
| semantic_retriever    | 1.0000    | 0.8695    | 0.6480    |0.9565    |0.2034    |0.2869    |

<br>

LangSmith Evaluators:

| Retriver | Correctness | Empathy |Helpfulness | Latency | Tokens |Cost
|-----------------|-----------------|-----------------|-----------------|-----------------|-----------------|-----------------|
| naive_retrieval    | 1.00   | 0.00    | 1.00    |5.239   |111,291.0    |0.0122   |
| bm25_retriever    | 1.00   | 0.00    | 1.00     |4.576    |50,477.0    |0.005    |
| compression_retriever   | 1.00   | 0.00    | 1.00     |5.216    |35,147.0    |0.0044    |
| multi_query_retriever    | 1.00   | 0.00    | 1.00    |8.837   |139,275.0    |0.0152    |
| parent_document_retriever    | 1.00   | 0.00    | 1.00    |4.572    |47,782.0    |0.0056    |
| ensemble_retriever    | 1.00   | 0.00    | 1.00     |11.03    |153,111.0    |0.0166    |
| semantic_retriever    | 1.00   | 0.00    | 1.00     |3.906    |102,019.0    |0.0112    |


Based on the comprehensive evaluation of retriever performance for financial analysis, wealth, and asset management data, the BM25 Retriever emerges as the optimal choice for this particular use case. While the Ensemble Retriever achieves the highest overall performance metrics (context_recall: 1.0000, faithfulness: 0.9633), its significantly higher cost ($0.0166 vs $0.005) and latency (11.03s vs 4.576s) make it prohibitively expensive for production deployment. The BM25 Retriever strikes an excellent balance across all critical factors: it maintains strong performance metrics (context_recall: 0.9667, faithfulness: 0.8398, answer_relevancy: 0.9562) while being the most cost-effective option at $0.005 per evaluation, consuming the fewest tokens (50,477), and delivering the second-fastest latency at 4.576 seconds. The Parent Document Retriever also shows competitive performance with similar cost and latency profiles, but BM25's superior faithfulness and answer relevancy scores make it the preferred choice for financial applications where accuracy and cost-efficiency are paramount. For organizations prioritizing budget-conscious deployment without sacrificing quality, BM25 provides the best value proposition, offering 80% cost savings compared to the Ensemble approach while maintaining 90% of the performance quality.